# 🚀 Train Virgo 1.0 (one click)

**Before you start (once):**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. On Hugging Face, open [google/gemma-3-4b-it](https://huggingface.co/google/gemma-3-4b-it) and click **Agree and access**.
3. Get a Hugging Face token: [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → **Create new token** → type **Write** → copy it.
4. The repo is private, so you'll also be asked for a **GitHub token** ([create one](https://github.com/settings/personal-access-tokens/new): only **virgo-models**, **Contents: Read-only**). Or make the repo public to skip this.

**Then:** menu **Runtime → Run all**. Paste your token when asked. That's it. Keep this tab open (~30–60 min).

Tip: set `QUICK_TEST = True` below for a fast first try (~10–15 min).

In [ ]:
#@title ⚙️ Settings
QUICK_TEST = False  #@param {type:"boolean"}
BASE_MODEL = "google/gemma-3-4b-it"  #@param ["google/gemma-3-4b-it", "google/gemma-3-1b-it"]
SAVE_TO_HUGGING_FACE = True  #@param {type:"boolean"}

In [ ]:
#@title 1️⃣ Set up (install + sign in)
import os, subprocess, sys, torch
if not torch.cuda.is_available():
    raise SystemExit("❌ No GPU. Menu Runtime → Change runtime type → T4 GPU, then Run all again.")
print("✅ GPU:", torch.cuda.get_device_name(0))
os.chdir("/content")
if not os.path.isdir("virgo-models/.git"):
    subprocess.run(["rm", "-rf", "virgo-models"])
    REPO = "github.com/nangmrr752/virgo-models"
    if subprocess.run(["git", "clone", "-q", f"https://{REPO}"]).returncode != 0:
        # The repo is private: sign in with a GitHub token (read access is enough).
        gh = None
        try:
            from google.colab import userdata
            gh = userdata.get("GITHUB_TOKEN")  # optional: save it once under 🔑 Secrets as GITHUB_TOKEN
        except Exception:
            pass
        if not gh:
            from getpass import getpass
            print("The repo is private. Create a GitHub token: github.com/settings/personal-access-tokens/new")
            print("→ Repository access: Only select repositories → virgo-models → Permissions: Contents = Read-only.")
            gh = getpass("Paste your GitHub token and press Enter: ").strip()
        if subprocess.run(["git", "clone", "-q", f"https://x-access-token:{gh}@{REPO}"]).returncode != 0:
            raise SystemExit("❌ Couldn't download virgo-models. Check the GitHub token has access to the repo.")
print("✅ Downloaded Virgo's code")
os.chdir("/content/virgo-models")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("✅ Installed")

from huggingface_hub import login
token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")  # optional: save it once under 🔑 Secrets as HF_TOKEN
except Exception:
    pass
if not token:
    from getpass import getpass
    token = getpass("Paste your Hugging Face token and press Enter: ").strip()
login(token=token)
print("✅ Signed in to Hugging Face")

In [ ]:
#@title 2️⃣ Train Virgo
epochs = "1" if QUICK_TEST else "3"
!python scripts/check_data.py
!python chat/train.py --base {BASE_MODEL} --epochs {epochs}

In [ ]:
#@title 3️⃣ Score Virgo
!python chat/evaluate.py --base {BASE_MODEL}

In [ ]:
#@title 4️⃣ Try Virgo
import sys; sys.path.append("chat")
from virgo_chat import VirgoChat
bot = VirgoChat(BASE_MODEL, adapter="chat/out/virgo-1.0-chat-lora")
for q in ["Who are you?", "អ្នកជានរណា?", "What is 15% of 200?", "ពន្យល់ AI ឱ្យងាយយល់"]:
    print("🧑", q)
    print("🤖", bot.reply([{"role": "user", "content": q}], temperature=0), "\n")

In [ ]:
#@title 5️⃣ Save Virgo to your Hugging Face account (private)
if SAVE_TO_HUGGING_FACE:
    from huggingface_hub import HfApi
    api = HfApi()
    repo = f"{api.whoami()['name']}/virgo-1.0-chat-lora"
    api.create_repo(repo, private=True, exist_ok=True)
    api.upload_folder(folder_path="chat/out/virgo-1.0-chat-lora", repo_id=repo)
    print(f"✅ Saved: https://huggingface.co/{repo}")
print("🎉 Done! Send the score table and Virgo's answers above to Claude.")